# Semantic menu enrichment

Generate guarded descriptions for sparse items, score a closed semantic vocabulary locally, and apply provenance-aware enrichment to a freshly normalized menu.

In [ ]:
%pip install -r requirements-enrichment.txt

In [ ]:
%run ./01_menu_api_ingestion.ipynb

In [4]:
import json
from pathlib import Path

from enrichment_cache import EnrichmentCache
from menu_descriptions import (
    GeminiDescriptionGenerator,
    apply_description_overrides,
    generate_descriptions,
    load_cached_descriptions,
    needs_description,
)
from menu_enrichment import StrandsDeciderScorer, apply_enrichment, score_menu
from menu_validation import validate_menu

validate_menu(menu).raise_for_errors()
cache_path = Path("enrichment") / f"{menu.id}.json"
cache = EnrichmentCache.load(cache_path, menu.id)
overrides_path = Path("enrichment/description_overrides.json")
if overrides_path.exists():
    override_count = apply_description_overrides(
        menu,
        cache,
        json.loads(overrides_path.read_text(encoding="utf-8")),
        cache_path=cache_path,
    )
    print(f"Applied {override_count} curated description overrides")
missing_count = sum(needs_description(item) for item in menu.items)
print(f"{missing_count} of {len(menu.items)} items need an inferred description")

Applied 0 curated description overrides
145 of 224 items need an inferred description


In [ ]:

description_generator = GeminiDescriptionGenerator()
description_results = generate_descriptions(
    menu, description_generator, cache, cache_path=cache_path, batch_size=25
)

described = [result for result in description_results if result.status == "described"]
for result in described[:10]:
    print(result.item_id, result.text)
for result in description_results:
    if result.status != "described":
        print(result.status.upper(), result.item_id, result.reason or "model returned unknown")

In [5]:
# Load the completed descriptions from disk; this does not call Gemini.
description_results = load_cached_descriptions(menu, cache, require_all=True)
print(f"Loaded {len(description_results)} cached descriptions")

# score_menu reuses current cached scores and calls Strands only for missing or stale items.
print(f"Cache contains {len(cache.tags)} score rows for {len(menu.items)} menu items")
scorer = StrandsDeciderScorer()
tag_results = score_menu(
    menu,
    description_results,
    scorer,
    cache,
    cache_path=cache_path,
    progress=lambda done, total: print(f"Scored missing/stale items {done}/{total}"),
)
print("Strands scores are current for all menu items.")

Loaded 145 cached descriptions


/Users/hassankhan/projects/business-projects/order-assistant/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 6595.66it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Checked semantic scores 25/224
Checked semantic scores 50/224
Checked semantic scores 75/224
Checked semantic scores 100/224
Checked semantic scores 125/224
Checked semantic scores 150/224
Checked semantic scores 175/224
Checked semantic scores 200/224
Checked semantic scores 224/224
Strands scores are current for all menu items.


In [ ]:
enriched_menu = apply_enrichment(menu, description_results, tag_results)
tag_by_id = {result.item_id: result for result in tag_results}

input_level_counts = {}
for item in enriched_menu.items:
    result = tag_by_id[item.id]
    input_level_counts[result.input_level] = input_level_counts.get(result.input_level, 0) + 1
    accepted = [value.value for value in item.semantic_attributes]
    scales = {name: round(score.value, 3) for name, score in result.scores.scales.items()}
    print(item.name, result.input_level, accepted, scales)
print("Input levels:", input_level_counts)

enriched_validation = validate_menu(enriched_menu)
print(f"Validation: {len(enriched_validation.errors)} error(s), {len(enriched_validation.warnings)} warning(s)")
enriched_validation.raise_for_errors()

## Input-level comparison

This fixed set compares semantic decisions at the source-description, inferred-description, name-and-category, and name-only levels. The last three deliberately vague names should produce few confident positive tags.

In [11]:
comparison_cases = [
    ("Gulab Jamun", "Desserts", "Fried milk dumplings in sugar syrup.", "A soft, syrup-soaked Indian sweet served as dessert.", {"sweet", "dessert", "rich"}),
    ("Lime Soda", "Drinks", "Chilled lime soda.", "A fizzy lime drink with a bright, cooling taste.", {"refreshing", "light"}),
    ("Chicken Vindaloo", "Curries", "Hot Goan-style chicken curry.", "A robust Goan-style curry typically known for heat and tang.", {"spicy", "savory", "filling"}),
    ("Plain Naan", "Breads", "Soft baked flatbread.", "A soft baked flatbread usually served alongside a main dish.", {"savory"}),
    ("Fruit Salad", "Desserts", "Mixed fresh fruit.", "A cool mixture of fresh fruit with juicy, crisp textures.", {"sweet", "light", "refreshing", "dessert"}),
    ("Full English Breakfast", "Breakfast", "Eggs, sausage, beans and toast.", "A substantial cooked breakfast with several savory components.", {"savory", "filling", "breakfast"}),
    ("House Special", "", "", "", set()),
    ("Meal Deal 2", "", "", "", set()),
    ("Chef's Choice", "", "", "", set()),
]

levels = {
    "source_description": lambda name, category, source, inferred: f"Name: {name}\nCategories: {category}\nDescription: {source}",
    "inferred_description": lambda name, category, source, inferred: f"Name: {name}\nCategories: {category}\nInferred description: {inferred}",
    "name_category": lambda name, category, source, inferred: f"Name: {name}\nCategories: {category}",
    "name_only": lambda name, category, source, inferred: f"Name: {name}",
}

for level, make_text in levels.items():
    correct = total = 0
    for name, category, source, inferred, expected in comparison_cases:
        scores = scorer.score(make_text(name, category, source, inferred))
        predicted = {tag for tag, probability in scores.tags.items() if probability >= 0.7}
        for tag in scores.tags:
            correct += (tag in predicted) == (tag in expected)
            total += 1
    print(level, f"{correct / total:.1%}")

source_description 91.9%
inferred_description 88.9%
name_category 92.9%
name_only 91.9%
